<a id="projectile-main"></a>
# 02-1. PhysicsNeMo 기본 — 발사체 운동 PINN

**세션:** 14:40–16:00 (PhysicsNeMo 기본 / PINN)  
**선행:** [GH200 GPU 메모리와 프로파일링](../01_GH200/02_GPU_Memory_Profile.ipynb)

**핵심 질문:** 정답 궤적을 학습 데이터로 주지 않아도, 초기조건과 운동방정식만으로 시간에 따른 위치를 구할 수 있을까요?

| 참가자가 직접 하는 일 | 확인할 결과 |
|---|---|
| 초기속도와 발사각 선택 | 예상 비행시간·도달거리·최고점 변화 |
| 해석해 함수의 두 줄 완성 | 운동방정식으로 계산한 기준값 |
| PINN 학습 조건과 코드 연결 | 초기조건 손실과 ODE 잔차 손실의 역할 |
| 학습 단계 수를 정하고 실행 | 학습 전 예상과 실제 오차 비교 |
| 특정 시점의 예측값 계산 | 학습 구간과 외삽 구간의 차이 해석 |

완료 기준은 `validator.npz`와 `inferencer_data.npz` 생성, 학습 구간과 외삽 구간의 오차 계산, 두 구간을 나누어 표시한 그래프 확인입니다.


## 실습 순서

1. 실행 환경과 지원 파일을 확인합니다.
2. 물리 실험값을 바꾸고 해석해 코드를 완성합니다.
3. PINN의 입력·출력·손실을 운동방정식과 연결합니다.
4. PhysicsNeMo-Sym 구성도와 실제 코드를 대응시킵니다.
5. 학습 단계 수를 정하고 새 결과 폴더에서 학습합니다.
6. 해석해와 PINN 예측을 비교하고 학습 구간과 외삽 구간의 오차 차이를 기록합니다.

`[직접 수정]` 표시가 있는 셀은 실행 전에 값을 바꾸거나 코드를 완성합니다. 오류가 나면 바로 아래의 정답 확인 상자를 참고한 뒤 다시 실행합니다.


## 1. 환경과 실습 폴더 확인

행사 환경에는 Linux/ARM64용 PhysicsNeMo 25.11과 실습 코드가 SIF 이미지에 포함되어 있습니다. 이 노트북에서는 패키지를 설치하거나 인터넷에서 파일을 내려받지 않습니다.


In [ ]:
from pathlib import Path
import platform
import subprocess
import sys
import time

launch_dir = Path.cwd().resolve()
REPO_ROOT = next(
    (
        candidate
        for candidate in (launch_dir, *launch_dir.parents)
        if (candidate / "labs" / "projectile" / "source_code" / "projectile.py").is_file()
    ),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError(
        "labs/projectile/source_code/projectile.py를 찾지 못했습니다. "
        "KSC2026 과정 폴더 안에서 노트북을 열었는지 확인하세요."
    )

LAB_DIR = REPO_ROOT / "labs" / "projectile"
SOURCE_DIR = LAB_DIR / "source_code"
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

required = [
    SOURCE_DIR / "projectile.py",
    SOURCE_DIR / "projectile_eqn.py",
    SOURCE_DIR / "conf" / "config.yaml",
    LAB_DIR / "images" / "projectile.svg",
    LAB_DIR / "images" / "physicsnemo_sym_workflow.webp",
]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError("필수 파일이 없습니다: {}".format(missing))

import torch
import physicsnemo
import physicsnemo.sym

print("과정 루트       : {}".format(REPO_ROOT))
print("발사체 실습 폴더: {}".format(LAB_DIR))
print("Python          : {}".format(sys.version.split()[0]))
print("시스템 아키텍처 : {}".format(platform.machine()))
print("PyTorch         : {}".format(torch.__version__))
print("PhysicsNeMo     : {}".format(getattr(physicsnemo, "__version__", "버전 정보 없음")))
print("CUDA 사용 가능  : {}".format(torch.cuda.is_available()))
if torch.cuda.is_available():
    print("GPU             : {}".format(torch.cuda.get_device_name(0)))
else:
    print("경고: CUDA GPU를 사용할 수 없습니다. 이 셀의 전체 출력을 강사에게 전달하세요.")


<a id="projectile-problem"></a>
## 2. 문제 설정 — 초기속도로 쏜 물체의 위치

원점에서 초기속도 `v₀`, 발사각 `θ`로 출발한 물체의 위치 `(x(t), y(t))`를 구합니다. 위쪽을 양의 `y` 방향으로 두며, 공기저항과 지면 충돌은 모델에 포함하지 않습니다.

<p align="center"><img src="../labs/projectile/images/projectile.svg" width="620" alt="발사체 운동의 좌표축, 초기속도와 포물선 궤적" /></p>

$$x''(t)=0, \qquad y''(t)=-g$$

$$x(t)=v_0\cos\theta\,t, \qquad y(t)=v_0\sin\theta\,t-\frac{1}{2}gt^2$$

해석해는 가중치 업데이트에 사용하지 않습니다. 학습 중 정해진 Validator 주기와 학습 종료 후에 PINN의 정확도를 확인하는 기준값으로 사용합니다.


In [ ]:
import numpy as np

# [직접 수정 1] 아래 범위에서 초기속도와 발사각을 선택합니다.
INITIAL_SPEED_M_S = 40.0       # 25.0 <= 값 <= 55.0
LAUNCH_ANGLE_DEG = 60.0       # 25.0 <= 값 <= 75.0
GRAVITY_M_S2 = 9.81

if not 25.0 <= INITIAL_SPEED_M_S <= 55.0:
    raise ValueError("INITIAL_SPEED_M_S는 25.0~55.0 범위에서 선택하세요.")
if not 25.0 <= LAUNCH_ANGLE_DEG <= 75.0:
    raise ValueError("LAUNCH_ANGLE_DEG는 25.0~75.0 범위에서 선택하세요.")

launch_angle_rad = np.deg2rad(LAUNCH_ANGLE_DEG)
velocity_x = INITIAL_SPEED_M_S * np.cos(launch_angle_rad)
velocity_y = INITIAL_SPEED_M_S * np.sin(launch_angle_rad)
expected_flight_time = 2.0 * velocity_y / GRAVITY_M_S2
expected_range = velocity_x * expected_flight_time
expected_apex = velocity_y**2 / (2.0 * GRAVITY_M_S2)

print("초기 속도 성분 : vx={:.2f} m/s, vy={:.2f} m/s".format(velocity_x, velocity_y))
print("예상 비행시간  : {:.2f} s".format(expected_flight_time))
print("예상 도달거리  : {:.2f} m".format(expected_range))
print("예상 최고점    : {:.2f} m".format(expected_apex))


### 활동 1 — 해석해 코드 두 줄 완성

위의 해석해 식을 NumPy 코드로 옮깁니다. 다음 셀의 `0.0` 두 곳을 각각 `x(t)`, `y(t)` 식으로 바꾼 뒤 실행합니다. 셀 끝의 검사는 네 개의 초기조건과 두 운동방정식이 모두 맞는지 수치적으로 확인합니다.


In [ ]:
def exact_position(t):
    t = np.asarray(t, dtype=np.float64)

    # [직접 수정 2] 아래 두 줄의 0.0을 해석해 식으로 바꿉니다.
    x = np.zeros_like(t)
    y = np.zeros_like(t)
    return x, y


check_times = np.array([0.0, 0.5, 1.0, 2.0])
x_check, y_check = exact_position(check_times)
if np.allclose(x_check, 0.0) or np.allclose(y_check, 0.0):
    raise AssertionError("아직 해석해 코드가 완성되지 않았습니다. x와 y 두 줄을 수정하세요.")

dt = 1.0e-3
x0, y0 = exact_position(0.0)
x_plus, y_plus = exact_position(dt)
x_minus, y_minus = exact_position(-dt)
vx_numeric = (x_plus - x_minus) / (2.0 * dt)
vy_numeric = (y_plus - y_minus) / (2.0 * dt)
ax_numeric = (x_plus - 2.0 * x0 + x_minus) / dt**2
ay_numeric = (y_plus - 2.0 * y0 + y_minus) / dt**2

assert np.isclose(x0, 0.0, atol=1.0e-10)
assert np.isclose(y0, 0.0, atol=1.0e-10)
assert np.isclose(vx_numeric, velocity_x, rtol=1.0e-5)
assert np.isclose(vy_numeric, velocity_y, rtol=1.0e-5)
assert np.isclose(ax_numeric, 0.0, atol=1.0e-6)
assert np.isclose(ay_numeric, -GRAVITY_M_S2, rtol=1.0e-5)
print("해석해 검사: PASS")


<details>
<summary><strong>정답 확인 · 해석해 코드</strong></summary>

```python
x = velocity_x * t
y = velocity_y * t - 0.5 * GRAVITY_M_S2 * t**2
```

두 줄을 위 코드로 바꾸면 위치와 속도의 초기조건, `x''=0`, `y''=-g`가 함께 성립합니다.
</details>


In [ ]:
import matplotlib.pyplot as plt

time_exact = np.linspace(0.0, expected_flight_time, 300)
x_exact, y_exact = exact_position(time_exact)

figure, axis = plt.subplots(figsize=(7.5, 4.2), constrained_layout=True)
axis.plot(x_exact, y_exact, color="#76B900", linewidth=2.5)
axis.scatter([x_exact[0], x_exact[-1]], [y_exact[0], y_exact[-1]], color="black", zorder=3)
axis.set(title="해석해로 구한 발사체 궤적", xlabel="x (m)", ylabel="y (m)")
axis.grid(alpha=0.25)
plt.show()


## 3. PINN의 입력·출력·손실

```text
t ──► 완전연결 신경망 ──► x̂(t), ŷ(t)
                              │
                              └─ 자동미분 ─► x̂'', ŷ''
```

신경망은 시간 `t`를 받아 위치 `x̂, ŷ`를 출력합니다. 자동미분으로 시간에 대한 도함수를 계산하고 다음 손실을 최소화합니다.

$$L=L_{IC}+L_{ODE}$$

- `L_IC`: `t=0`에서 `x=y=0`, `x'=v₀cosθ`, `y'=v₀sinθ`
- `L_ODE`: `r_x=x̂''=0`, `r_y=ŷ''+g=0`

두 식이 2계 ODE이므로 위치 두 개와 속도 두 개, 모두 네 개의 초기조건이 필요합니다. 궤적의 정답 좌표는 학습 손실에 포함되지 않습니다.


<a id="physicsnemo-workflow"></a>
## 4. PhysicsNeMo-Sym 학습 구성과 코드 대응

<p align="center"><img src="../labs/projectile/images/physicsnemo_sym_workflow.webp" width="900" alt="PhysicsNeMo-Sym에서 문제를 구성하고 학습을 실행하는 절차" /></p>

**그림의 범위:** PhysicsNeMo-Sym에서 설정·방정식·모델·학습 조건·평가 항목을 `Domain`에 등록하고 `Solver`를 실행하는 공통 구성 흐름입니다. 발사체 PINN의 코드 대응은 다음 표와 같습니다. FNO 모델 내부의 데이터 흐름은 두 번째 노트북에서 별도로 다룹니다.

| 그림의 라벨 | 이 실습의 코드 | 역할 |
|---|---|---|
| Load Hydra | `source_code/conf/config.yaml` | 신경망·최적화 방법·학습 단계 설정 |
| Define Geometry | `Point1D(0)` + 시간 변수 `t` | Constraint API의 고정점과 실제 입력 시간 샘플링 |
| Load Datasets | 사용하지 않음 | PINN 학습에는 정답 궤적 데이터셋이 필요하지 않음 |
| Create Nodes | `ProjectileEquation` + `projectile_net` | `t → x̂,ŷ`와 자동미분 잔차 계산 |
| Create Constraint | `initial_condition`, `ode_constraint` | 초기조건과 운동방정식을 학습 손실로 등록 |
| Create Validator | `validator` | `0≤t<5`에서 해석해와 예측 비교 |
| Create Inferencer | `grid_inference` | `0≤t<8`에서 정답 없이 예측 생성 |
| Create Monitor | 사용하지 않음 | 이 실습에서는 별도 모니터를 등록하지 않음 |
| Create Domain | `projectile_domain` | Constraint 2개, Validator 1개, Inferencer 1개 등록 |
| Create/Run Solver | `Solver(cfg, projectile_domain).solve()` | 등록한 구성으로 학습 실행 |

그림의 `N_c`, `N_v`, `N_i`, `N_m`은 각각 등록한 Constraint, Validator, Inferencer, Monitor의 개수입니다. 이 실습에서는 `N_c=2`, `N_v=1`, `N_i=1`, `N_m=0`입니다.


## 5. 방정식 노드와 학습 조건

`ProjectileEquation`은 운동방정식을 잔차가 0이 되는 형태로 정의합니다.

```python
self.equations = {
    "ode_x": x.diff(t, 2),
    "ode_y": y.diff(t, 2) + gravity,
}
```

따라서 ODE Constraint의 목표값은 `{"ode_x": 0.0, "ode_y": 0.0}`입니다. `ode_y`에 중력항이 이미 포함되어 있으므로 목표값을 `-9.81`로 두지 않습니다.


In [ ]:
from projectile_eqn import ProjectileEquation

equation = ProjectileEquation(gravity=GRAVITY_M_S2)
print("등록한 방정식 잔차:")
for name, expression in equation.equations.items():
    print("  {}: {} = 0".format(name, expression))


### 활동 2 — ODE 잔차의 목표값 바로잡기

`ProjectileEquation`의 `ode_y`는 `y''+g`까지 포함한 **잔차 이름**입니다. 아래 사전의 `ode_y` 목표값을 잔차가 0이 되도록 고치십시오. 이는 중력가속도 자체를 없앤다는 뜻이 아니라, 신경망 예측이 `y''+g=0`을 만족하도록 학습한다는 뜻입니다.


In [ ]:
# [직접 수정 3] ode_y 잔차의 올바른 목표값으로 바꿉니다.
RESIDUAL_TARGETS = {"ode_x": 0.0, "ode_y": -GRAVITY_M_S2}

if RESIDUAL_TARGETS != {"ode_x": 0.0, "ode_y": 0.0}:
    raise AssertionError(
        "ProjectileEquation이 ode_y = y'' + g로 정의되어 있습니다. "
        "따라서 ode_y Constraint의 목표값을 다시 확인하세요."
    )
print("ODE 잔차 목표값 검사: PASS")


`Point1D(0)`은 궤적을 나타내는 1차원 기하 형상이 아니라, PhysicsNeMo-Sym Constraint API가 요구하는 **샘플링 기준점**입니다. 신경망의 실제 입력 `t`는 `parameterization={t: ...}`에서 생성됩니다.

| 구성 요소 | 가중치 업데이트 | 필요한 기준값 | 역할 |
|---|---:|---|---|
| 초기조건 Constraint | 사용 | 시작 위치·속도 | 해의 출발 조건 고정 |
| ODE Constraint | 사용 | 잔차 목표값 0 | 운동방정식 만족 |
| Validator | 사용하지 않음 | 해석해 | 학습 구간의 정확도 평가 |
| Inferencer | 사용하지 않음 | 없음 | 더 넓은 시간 구간의 예측 저장 |


## 6. 실제 설정을 읽고 학습 실험값 정하기

설정 파일을 직접 읽어 신경망과 학습 조건을 확인합니다. 이어지는 셀에서 선택한 초기속도·발사각·학습 단계 수는 Hydra override로 학습 스크립트에 전달됩니다.


In [ ]:
import yaml

config_path = SOURCE_DIR / "conf" / "config.yaml"
config = yaml.safe_load(config_path.read_text(encoding="utf-8"))

# [직접 수정 4] 수업 기본값 5000을 유지하거나 강사 안내 범위에서 바꿉니다.
MAX_STEPS = int(config["training"]["max_steps"])
TRAIN_TIME_END_S = float(config["custom"]["train_time_end"])
INFERENCE_TIME_END_S = float(config["custom"]["inference_time_end"])
if MAX_STEPS < 1000:
    raise ValueError("수업 실습에서는 MAX_STEPS를 1000 이상으로 지정하세요.")
if not 0.0 < TRAIN_TIME_END_S < INFERENCE_TIME_END_S:
    raise ValueError("학습 종료 시점은 0보다 크고 추론 종료 시점보다 작아야 합니다.")

print("설정 파일       : {}".format(config_path))
print("최대 학습 단계  : {}".format(MAX_STEPS))
print("최종 검증·추론 저장 단계: {}".format(MAX_STEPS))
print("학습 시간 구간  : 0.0 <= t < {:.1f} s".format(TRAIN_TIME_END_S))
print("전체 추론 구간  : 0.0 <= t < {:.1f} s".format(INFERENCE_TIME_END_S))
print("초기조건 배치   : {}".format(config["batch_size"]["initial_x"]))
print("ODE 내부점 배치 : {}".format(config["batch_size"]["interior"]))
print("실험 초기속도   : {:.1f} m/s".format(INITIAL_SPEED_M_S))
print("실험 발사각     : {:.1f} deg".format(LAUNCH_ANGLE_DEG))


## 7. 학습 전 예상과 Solver 실행

실행 전에 예상 결과를 한 문장으로 적습니다. 예: “학습 구간에서는 두 곡선이 겹치지만, 외삽 구간에서는 오차가 커질 것이다.”


In [ ]:
# [직접 수정 5] 실행 전에 자신의 예상을 적습니다.
HYPOTHESIS = "학습 구간과 외삽 구간에서 예측 오차가 어떻게 달라질지 적으세요."
print("학습 전 예상:", HYPOTHESIS)


기본 실행은 새 결과 폴더를 만듭니다. 중단된 학습을 이어갈 때만 강사가 확인한 기존 결과 폴더를 `RESUME_RUN_DIR`에 지정합니다. 자동으로 예전 체크포인트를 선택하지 않으므로 다른 실험의 결과가 섞이지 않습니다.


In [ ]:
from datetime import datetime

RESUME_RUN_DIR = None  # 재개할 때만 기존 run 폴더의 절대 경로를 지정합니다.

if RESUME_RUN_DIR is None:
    RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    RUN_OUTPUT_DIR = LAB_DIR / "outputs" / "ksc_projectile" / RUN_ID
    RUN_MODE = "새 학습"
else:
    RUN_OUTPUT_DIR = Path(RESUME_RUN_DIR).expanduser().resolve()
    if not RUN_OUTPUT_DIR.is_dir():
        raise FileNotFoundError("재개할 run 폴더가 없습니다: {}".format(RUN_OUTPUT_DIR))
    RUN_MODE = "학습 재개"

command = [
    sys.executable,
    "source_code/projectile.py",
    "network_dir={}".format(RUN_OUTPUT_DIR),
    "training.max_steps={}".format(MAX_STEPS),
    "training.rec_validation_freq={}".format(MAX_STEPS),
    "training.rec_inference_freq={}".format(MAX_STEPS),
    "custom.gravity={}".format(GRAVITY_M_S2),
    "custom.initial_speed={}".format(INITIAL_SPEED_M_S),
    "custom.launch_angle_deg={}".format(LAUNCH_ANGLE_DEG),
]
print("실행 방식 : {}".format(RUN_MODE))
print("결과 폴더 : {}".format(RUN_OUTPUT_DIR))
print("실행 명령 : {}".format(" ".join(str(part) for part in command)))

started = time.perf_counter()
completed = subprocess.run(command, cwd=LAB_DIR)
TRAIN_WALL_SECONDS = time.perf_counter() - started

print("전체 실행 시간: {:.2f} min".format(TRAIN_WALL_SECONDS / 60.0))
if completed.returncode != 0:
    raise RuntimeError("Projectile 학습 실패 (exit code={})".format(completed.returncode))

validator_path = RUN_OUTPUT_DIR / "validators" / "validator.npz"
inferencer_path = RUN_OUTPUT_DIR / "inferencers" / "inferencer_data.npz"
if not validator_path.is_file():
    raise FileNotFoundError("검증 결과를 찾지 못했습니다: {}".format(validator_path))
if not inferencer_path.is_file():
    raise FileNotFoundError("추론 결과를 찾지 못했습니다: {}".format(inferencer_path))
print("검증 결과: {}".format(validator_path))


## 8. 해석해와 PINN 예측 비교

Validator가 저장한 `0≤t<5 s` 결과에서 상대 L2 오차와 최대 절대 오차를 계산합니다. 상대 L2 오차는 전체 곡선의 오차 크기를 기준 곡선의 크기로 나눈 값입니다.


In [ ]:
payload = np.load(validator_path, allow_pickle=True)
validation = np.atleast_1d(payload.f.arr_0)[0]

t_validation = validation["t"][:, 0]
x_true = validation["true_x"][:, 0]
x_pred = validation["pred_x"][:, 0]
y_true = validation["true_y"][:, 0]
y_pred = validation["pred_y"][:, 0]

def relative_l2(prediction, target):
    return np.linalg.norm(prediction - target) / np.linalg.norm(target)

metrics = {
    "x_relative_l2": relative_l2(x_pred, x_true),
    "y_relative_l2": relative_l2(y_pred, y_true),
    "x_max_abs_error_m": np.max(np.abs(x_pred - x_true)),
    "y_max_abs_error_m": np.max(np.abs(y_pred - y_true)),
}
for name, value in metrics.items():
    print("{:22s}: {:.6e}".format(name, value))

figure, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
axes[0].plot(t_validation, x_true, label="x(t) 해석해")
axes[0].plot(t_validation, x_pred, "--", label="x(t) PINN")
axes[0].set(title="수평 위치", xlabel="t (s)", ylabel="x (m)")
axes[0].legend()
axes[0].grid(alpha=0.2)

axes[1].plot(t_validation, y_true, label="y(t) 해석해")
axes[1].plot(t_validation, y_pred, "--", label="y(t) PINN")
axes[1].set(title="수직 위치", xlabel="t (s)", ylabel="y (m)")
axes[1].legend()
axes[1].grid(alpha=0.2)

prediction_path = RUN_OUTPUT_DIR / "prediction.png"
figure.savefig(prediction_path, dpi=150, bbox_inches="tight")
plt.show()
print("현재 실행 그래프 저장: {}".format(prediction_path))


## 9. 학습 구간과 외삽 구간 비교

Inferencer는 정답을 입력받지 않고 `0≤t<8 s`에서 PINN 예측 `x(t), y(t)`를 저장합니다. `0≤t<5 s`는 ODE Constraint가 시간을 샘플링한 **학습 구간**, `5≤t<8 s`는 학습 범위를 벗어난 **외삽 구간**입니다. 아래 셀은 두 구간의 오차를 따로 계산하고 배경색으로 구분합니다.

학습을 아직 실행하지 않았다면 결과 파일 경로와 먼저 실행할 셀을 안내하는 오류가 표시됩니다. 7절의 Solver 실행을 완료한 뒤 이 셀을 다시 실행하십시오.


In [ ]:
if "RUN_OUTPUT_DIR" not in globals():
    raise RuntimeError("먼저 7절의 Solver 실행 셀을 완료한 뒤 이 셀을 실행하세요.")

inferencer_path = RUN_OUTPUT_DIR / "inferencers" / "inferencer_data.npz"
if not inferencer_path.is_file():
    raise FileNotFoundError(
        "Inferencer 결과가 없습니다: {}. 먼저 7절의 Solver 실행 셀을 완료하세요.".format(
            inferencer_path
        )
    )

with np.load(inferencer_path, allow_pickle=True) as inferencer_payload:
    if "arr_0" not in inferencer_payload.files:
        raise KeyError("Inferencer NPZ에 arr_0 payload가 없습니다: {}".format(inferencer_path))
    inference = np.atleast_1d(inferencer_payload["arr_0"])[0]

if not isinstance(inference, dict):
    raise TypeError("Inferencer payload가 사전 형식이 아닙니다: {}".format(type(inference).__name__))
required_keys = {"t", "x", "y"}
missing_keys = sorted(required_keys.difference(inference))
if missing_keys:
    raise KeyError("Inferencer 결과에 필요한 키가 없습니다: {}".format(missing_keys))

t_inference = np.asarray(inference["t"], dtype=np.float64).reshape(-1)
x_inference = np.asarray(inference["x"], dtype=np.float64).reshape(-1)
y_inference = np.asarray(inference["y"], dtype=np.float64).reshape(-1)
if not (len(t_inference) == len(x_inference) == len(y_inference)):
    raise ValueError("Inferencer의 t, x, y 배열 길이가 서로 다릅니다.")
if len(t_inference) == 0 or not all(
    np.all(np.isfinite(values)) for values in (t_inference, x_inference, y_inference)
):
    raise ValueError("Inferencer 결과가 비어 있거나 유한하지 않은 값을 포함합니다.")

sort_order = np.argsort(t_inference)
t_inference = t_inference[sort_order]
x_inference = x_inference[sort_order]
y_inference = y_inference[sort_order]
x_inference_true, y_inference_true = exact_position(t_inference)
training_mask = (0.0 <= t_inference) & (t_inference < TRAIN_TIME_END_S)
extrapolation_mask = (TRAIN_TIME_END_S <= t_inference) & (t_inference < INFERENCE_TIME_END_S)
if not np.any(training_mask) or not np.any(extrapolation_mask):
    raise ValueError("Inferencer 결과에 학습 구간과 외삽 구간 표본이 모두 필요합니다.")

interval_metrics = {}
for label, mask in (("학습 구간", training_mask), ("외삽 구간", extrapolation_mask)):
    interval_metrics[label] = {
        "x_relative_l2": relative_l2(x_inference[mask], x_inference_true[mask]),
        "y_relative_l2": relative_l2(y_inference[mask], y_inference_true[mask]),
        "x_max_abs_error_m": np.max(np.abs(x_inference[mask] - x_inference_true[mask])),
        "y_max_abs_error_m": np.max(np.abs(y_inference[mask] - y_inference_true[mask])),
    }

for label, values in interval_metrics.items():
    print("\n{}".format(label))
    for name, value in values.items():
        print("  {:22s}: {:.6e}".format(name, value))

figure, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
for axis, truth, prediction, coordinate in (
    (axes[0], x_inference_true, x_inference, "x"),
    (axes[1], y_inference_true, y_inference, "y"),
):
    axis.axvspan(0.0, TRAIN_TIME_END_S, color="#76B900", alpha=0.10, label="학습 구간")
    axis.axvspan(
        TRAIN_TIME_END_S, INFERENCE_TIME_END_S, color="#F5A623", alpha=0.14, label="외삽 구간"
    )
    axis.plot(t_inference, truth, color="black", linewidth=2.0, label="{}(t) 해석해".format(coordinate))
    axis.plot(t_inference, prediction, "--", color="#1F77B4", label="{}(t) PINN".format(coordinate))
    axis.axvline(TRAIN_TIME_END_S, color="#A15C00", linewidth=1.0)
    axis.set(title="{} 위치: 학습과 외삽".format(coordinate), xlabel="t (s)", ylabel="{} (m)".format(coordinate))
    axis.grid(alpha=0.2)
    axis.legend(fontsize=8)

extrapolation_path = RUN_OUTPUT_DIR / "extrapolation.png"
figure.savefig(extrapolation_path, dpi=150, bbox_inches="tight")
plt.show()
print("학습·외삽 비교 그래프 저장: {}".format(extrapolation_path))


### 활동 3 — 학습 구간 안팎의 특정 시점 비교

`CHECK_TIME_S`를 먼저 학습 구간의 값으로, 다음에는 외삽 구간의 값으로 바꾸어 실행합니다. Inferencer 격자에서 가장 가까운 시점의 PINN 예측과 해석해를 비교하고 현재 시점이 어느 구간인지 표시합니다.


In [ ]:
# [직접 수정 6] 0.0 이상 8.0 미만에서 학습 구간과 외삽 구간의 시점을 각각 선택합니다.
CHECK_TIME_S = 6.50
if not 0.0 <= CHECK_TIME_S < INFERENCE_TIME_END_S:
    raise ValueError("CHECK_TIME_S는 Inferencer 범위인 0.0 이상 {:.1f} 미만이어야 합니다.".format(INFERENCE_TIME_END_S))

nearest_index = int(np.argmin(np.abs(t_inference - CHECK_TIME_S)))
t_nearest = float(t_inference[nearest_index])
x_exact_point, y_exact_point = exact_position(t_nearest)
interval_label = "학습 구간" if t_nearest < TRAIN_TIME_END_S else "외삽 구간"

print("선택 시점(최근접 격자): {:.3f} s ({})".format(t_nearest, interval_label))
print("x: exact={:.4f} m, PINN={:.4f} m, abs error={:.4e} m".format(
    float(x_exact_point), float(x_inference[nearest_index]),
    abs(float(x_inference[nearest_index]) - float(x_exact_point)),
))
print("y: exact={:.4f} m, PINN={:.4f} m, abs error={:.4e} m".format(
    float(y_exact_point), float(y_inference[nearest_index]),
    abs(float(y_inference[nearest_index]) - float(y_exact_point)),
))


## 10. 결과 해석

- Validator는 학습 범위인 `0≤t<5 s`에서 정확도를 평가합니다.
- Inferencer의 `0≤t<8 s` 중 `5–8 s`는 학습 구간 밖의 예측입니다.
- 물체가 지면으로 돌아온 뒤의 `y<0` 값은 지면 충돌을 포함하지 않은 운동방정식을 그대로 연장한 결과입니다.
- 이 모델은 선택한 초기속도와 발사각 한 조건의 궤적을 근사합니다. 초기조건 자체를 입력으로 받는 모델은 별도의 문제 설정이 필요합니다.

**기록할 내용**

1. 선택한 초기속도·발사각과 예상 비행시간을 적습니다.
2. 학습 구간과 외삽 구간의 상대 L2 오차를 비교합니다. 어느 좌표에서 차이가 더 큰지도 적습니다.
3. 두 배경색으로 나눈 그래프가 학습 전 예상과 일치했는지 설명합니다.
4. 공기저항이나 지면 충돌을 추가한다면 어느 방정식 또는 조건이 바뀌어야 하는지 적습니다.


## 11. 다음 단계 — 한 번의 발사 조건을 푸는 PINN에서 여러 소스항을 처리하는 FNO로

| 구분 | 발사체 운동 PINN | Poisson FNO |
|---|---|---|
| 입력 | 시간 `t` | 격자 위 소스항 `f(x,y)` |
| 출력 | 선택한 초기조건의 `x(t), y(t)` | 소스항에 대응하는 `u(x,y)` |
| 학습 신호 | 초기조건 + ODE 잔차 | 여러 소스항과 해의 쌍 `(f,u)` |
| 학습 후 질문 | 같은 조건에서 특정 시점의 위치는? | 처음 보는 소스항의 전체 해는? |

다음 [02_Poisson_FNO.ipynb](02_Poisson_FNO.ipynb)에서는 푸리에 공간의 직접해 코드를 완성한 뒤, FNO가 여러 입력장과 출력장의 관계를 학습하도록 구성합니다.


---

## 참고 자료와 라이선스

공식 문서와 논문 링크는 [PhysicsNeMo 모듈 안내](README.md#참고-자료)에 모았습니다. 노트북 실행에는 인터넷 연결이 필요하지 않습니다.

Copyright © 2026 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). Existing file-level notices remain in effect.
